In [1]:
# imports
import tensorflow as tf
from tensorflow import keras as keras
import matplotlib.pyplot as plt
import keras_tuner as kt

#tf.keras.utils.set_random_seed(1032)
print(tf.config.list_physical_devices('GPU'))

2024-05-21 13:50:34.982784: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-21 13:50:34.982838: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-21 13:50:34.983738: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-21 13:50:34.989267: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-21 13:50:35.583712: W tensorflow/compiler/tf2

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-05-21 13:50:36.248973: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 13:50:36.273399: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 13:50:36.273457: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
#import  data

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/test",
    shuffle = True,
    image_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/valid",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

data_augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
]

def  data_augmentation(images):
    for layer in  data_augmentation_layers:
        images = layer(images)
    return images

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize pixel values to [0, 1]
    image = data_augmentation(image)
    return image, label

train_data = train_data.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_data = val_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

test_data = test_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)


Found 2339 files belonging to 10 classes.


2024-05-21 13:50:36.438836: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 13:50:36.438898: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 13:50:36.438955: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 13:50:36.533191: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 13:50:36.533252: I external/local_xla/xla/stream_executor

Found 50 files belonging to 10 classes.
Found 50 files belonging to 10 classes.


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS =3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def Model(input_shape):

    model = keras.Sequential([
    keras.layers.Conv2D(32, kernel_size = (3, 3), activation='relu', input_shape=input_shape),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.BatchNormalization(),
    
    keras.layers.Conv2D(64, kernel_size=(3,3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.BatchNormalization(),
    
    
    keras.layers.Conv2D(96, kernel_size=(3,3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.BatchNormalization(),
    
    keras.layers.Conv2D(96, kernel_size=(3,3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Conv2D(64, kernel_size=(3,3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation = 'softmax')
    ])
    
    #, activity_regularizer = tf.keras.regularizers.L2()
    #, activity_regularizer = tf.keras.regularizers.L2(0.008)
    
    return model

Model(input_shape)

In [4]:
model = Model(input_shape)
model.compile(optimizer = keras.optimizers.Adam(learning_rate = 0.0005),
             loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
             metrics=['accuracy'])

In [5]:
#earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)

# min_delta=0.001
#,callbacks=[earlystopping]
history = model.fit(
train_data,
epochs = 50,
batch_size = 32,
validation_data = val_data)

Epoch 1/50


2024-05-21 13:50:50.134703: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_1/dropout_4/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2024-05-21 13:50:54.753624: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-05-21 13:50:55.894795: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-05-21 13:50:57.948072: I external/local_xla/xla/service/service.cc:168] XLA service 0x7ff760339180 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-05-21 13:50:57.948105: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2024-05-21 13:50:58.006772: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling ML

74/74 [==============================] - 19s 78ms/step - loss: 2.1328 - accuracy: 0.2612 - val_loss: 2.9706 - val_accuracy: 0.1200
Epoch 2/50
74/74 [==============================] - 4s 53ms/step - loss: 1.7603 - accuracy: 0.3596 - val_loss: 4.1511 - val_accuracy: 0.0800
Epoch 3/50
74/74 [==============================] - 4s 52ms/step - loss: 1.5989 - accuracy: 0.4241 - val_loss: 3.4384 - val_accuracy: 0.1000
Epoch 4/50
74/74 [==============================] - 4s 52ms/step - loss: 1.4362 - accuracy: 0.4716 - val_loss: 3.5816 - val_accuracy: 0.1200
Epoch 5/50
74/74 [==============================] - 4s 53ms/step - loss: 1.3518 - accuracy: 0.4853 - val_loss: 2.7525 - val_accuracy: 0.1600
Epoch 6/50
74/74 [==============================] - 4s 52ms/step - loss: 1.3107 - accuracy: 0.5246 - val_loss: 2.1610 - val_accuracy: 0.2800
Epoch 7/50
74/74 [==============================] - 4s 52ms/step - loss: 1.2436 - accuracy: 0.5511 - val_loss: 1.7980 - val_accuracy: 0.3200
Epoch 8/50
74/74 [=====